# 02 -- Classical Models + Language Model Feature (Step 2)Loads `cleaned_bfrd.csv` from Drive (produced by notebook 01), trains Logistic Regression / Naive Bayes / SVM with a TF-IDF + n-gram language model perplexity feature.

In [ ]:
!pip install -q nltk scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/BanglaFakeReviewDetector'
os.chdir(PROJECT_DIR)

import pandas as pd
df = pd.read_csv("cleaned_bfrd.csv", encoding="utf-8-sig")
counts = df["label"].value_counts()
AUTHENTIC_LABEL, FAKE_LABEL = counts.idxmax(), counts.idxmin()
print(f"Loaded {len(df)} rows. authentic={AUTHENTIC_LABEL!r}, fake={FAKE_LABEL!r}")

In [ ]:
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

BANGLA_STOPWORDS = set(stopwords.words('bengali'))
_porter = PorterStemmer()

EMOJI_PATTERN = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF" "\U00002190-\U000021FF" "]+", flags=re.UNICODE)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = re.sub(r"([\u0964!?.\-_=~*])\1{1,}", r"\1", text)
    text = re.sub(r"(#\w+)(\s*\1){1,}", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

def is_bangla_token(token):
    if not token:
        return False
    bangla_chars = sum(1 for ch in token if "\u0980" <= ch <= "\u09FF")
    return bangla_chars > len(token) / 2

def basic_tokenize(text):
    return re.findall(r"[\u0980-\u09FFa-zA-Z0-9]+", text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in BANGLA_STOPWORDS]

def stem_latin_tokens(tokens):
    return [_porter.stem(t) if not is_bangla_token(t) else t for t in tokens]

def preprocess_review(raw_text):
    cleaned = clean_text(raw_text)
    tokens = basic_tokenize(cleaned)
    return {"cleaned_text": cleaned, "tokens": stem_latin_tokens(remove_stopwords(tokens))}

print(f"Loaded {len(BANGLA_STOPWORDS)} Bengali stopwords.")

## N-gram language model perplexity feature

In [ ]:
from nltk.lm import Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.util import ngrams
import numpy as np

def train_language_model(texts, n=2):
    tokenized = [preprocess_review(t)["tokens"] for t in texts]
    train_data, vocab = padded_everygram_pipeline(n, tokenized)
    lm = Laplace(n)
    lm.fit(train_data, vocab)
    return lm

def perplexity_score(lm, text, n=2):
    tokens = preprocess_review(text)["tokens"]
    if not tokens:
        return float("inf")
    test_ngrams = list(ngrams(tokens, n, pad_left=True, pad_right=True, left_pad_symbol="<s>", right_pad_symbol="</s>"))
    return lm.perplexity(test_ngrams)

print("Language model functions ready.")

## Train/test split, TF-IDF + LM feature, train and evaluate

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from scipy.sparse import hstack, csr_matrix

train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df["label"])
train_df, test_df = df.loc[train_idx].copy(), df.loc[test_idx].copy()

authentic_train_texts = train_df.loc[train_df["label"] == AUTHENTIC_LABEL, "text"].tolist()
lm = train_language_model(authentic_train_texts)

for split_df in (train_df, test_df):
    split_df["lm_perplexity"] = split_df["text"].apply(lambda t: perplexity_score(lm, t))
    finite = split_df.loc[np.isfinite(split_df["lm_perplexity"]), "lm_perplexity"]
    cap = finite.max() if len(finite) else 1.0
    split_df["lm_perplexity"] = split_df["lm_perplexity"].replace([np.inf, -np.inf], cap)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(train_df["cleaned_text"])
X_test_tfidf = vectorizer.transform(test_df["cleaned_text"])

scaler = MinMaxScaler()
train_perp = scaler.fit_transform(train_df[["lm_perplexity"]])
test_perp = scaler.transform(test_df[["lm_perplexity"]])

X_train = hstack([X_train_tfidf, csr_matrix(train_perp)])
X_test = hstack([X_test_tfidf, csr_matrix(test_perp)])
y_train, y_test = train_df["label"], test_df["label"]

for name, model in [
    ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ("Naive Bayes", MultinomialNB()),
    ("Linear SVM", LinearSVC(class_weight="balanced", random_state=42)),
]:
    model.fit(X_train, y_train)
    print(f"--- {name} ---")
    print(classification_report(y_test, model.predict(X_test), zero_division=0))